## CGPA prediction model (attendance + payments + sponsorship + past performance)

Target: `CGPA` (semester-level cumulative GPA) from `student_transcript_*`.

**Input:** The modeling table from `02_integration_feature_engineering.ipynb`, which may include optional columns from Excel (or CSV) sources: students list (`TOTAL_REGISTRATIONS`, `STUDENT_LIST` for holdout), academic progression (`progression_count`, `retake_count_semester`, `max_attempt_semester`), and high school attributes. These are used when present for better prediction and evaluation.

Key modeling constraints:
- Avoid leakage: use lagged performance (`prev_*`) and same-semester attendance/payments as *predictors* for current-semester CGPA.
- Prevent student overlap between train/test where possible.

We train multiple models and select the best based on cross-validated error (MAE/RMSE) and a holdout evaluation.

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "outputs"

model_path = OUT_DIR / "modeling_table_student_semester.csv"
df = pd.read_csv(model_path)

df.shape

(55779, 74)

In [ ]:
TARGET = "CGPA"
GROUP = "REG_NO"

# Optional: if STUDENT_LIST exists (from high school join), we can do List15->train, List16->test.
has_student_list = "STUDENT_LIST" in df.columns
has_student_list

True

In [ ]:
# Build feature set

leak_cols = {
    # Anything that is mechanically derived from CGPA itself or cumulative totals in the same row should be treated carefully.
    # We KEEP current-semester attendance/payments and lagged performance; we DROP current cumulative GPA components that trivially reconstruct CGPA.
    "CUM_QUALITY_POINTS",
    "CUM_CREDITS_ATTEMPTED",
    "CUM_CREDITS_PASSED",
    "QUALITY_POINTS",  # current-semester QP relates strongly to GPA; keep or drop depending on interpretation
}

# Keep a conservative, interpretable set focused on trends + lagged performance.
base_keep = [
    "PROGRAM",
    "ACADEMIC_YEAR",
    "SEMESTER",
    "SEMESTER_INDEX",
    # Attendance
    "attendance_days","present_rate","late_rate","absent_rate","attendance_span_days",
    # Payments
    "payment_txn_count","payment_success_rate","total_paid_success","distinct_methods","distinct_channels","payment_span_days",
    # Sponsorship
    "has_sponsorship","sponsor_amount_total","sponsor_approved_count","sponsor_distinct_sponsors","sponsor_distinct_types",
    # Past performance (lagged)
    "prev_cgpa","prev_semester_gpa","prev_failed_courses","prev_passed_courses",
    "prev_fcw_count","prev_fex_count","prev_mex_count",
    "prev_credits_failed","prev_credits_passed",
    "prev_cum_credits_attempted","prev_cum_credits_passed",
]

# Optional enrichment if present (high school, students list, academic progression from 02)
optional = [c for c in ["SCHOOL_TIER","OWNERSHIP","DISTRICT","TOTAL_REGISTRATIONS","progression_count","retake_count_semester","max_attempt_semester"] if c in df.columns]
features = [c for c in base_keep if c in df.columns] + optional

# Drop rows without target
data = df.dropna(subset=[TARGET]).copy()

X = data[features].copy()
y = data[TARGET].astype(float)
groups = data[GROUP]

X.shape, y.shape

((55779, 37), (55779,))

In [ ]:
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# OneHotEncoder must output dense (sparse_output=False) so HistGradientBoostingRegressor works.
# If you get InvalidParameterError, use sparse=False instead (older sklearn).
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat_cols,
        ),
    ],
    remainder="drop",
)

# Slightly smaller models/folds for faster runs; increase n_estimators/max_iter/n_splits for final reporting.
models = {
    "ridge": Ridge(alpha=10.0, random_state=0),
    "rf": RandomForestRegressor(
        n_estimators=150,
        random_state=0,
        n_jobs=-1,
        min_samples_leaf=5,
    ),
    "hgb": HistGradientBoostingRegressor(
        random_state=0,
        max_depth=6,
        learning_rate=0.05,
        max_iter=200,
    ),
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv = GroupKFold(n_splits=3)

results = []
for name, est in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", est),
    ])
    cvres = cross_validate(pipe, X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": name,
        "mae": -float(np.mean(cvres["test_mae"])),
        "rmse": -float(np.mean(cvres["test_rmse"])),
        "r2": float(np.mean(cvres["test_r2"])),
    })

pd.DataFrame(results).sort_values("rmse")

,model,mae,rmse,r2
2,hgb,0.228710,0.377661,0.638998
1,rf,0.231593,0.384213,0.626353
0,ridge,0.243557,0.394261,0.606696


## Holdout evaluation (List16 holdout when available)

If `STUDENT_LIST` exists (via `student_high_schools_all.csv`), we evaluate on **List16** as a realistic out-of-sample cohort.

If it’s not available, we fall back to a group-based split by student.

In [ ]:
def evaluate_holdout(pipe: Pipeline, X_train, y_train, X_test, y_test) -> dict:
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    return {
        "mae": float(mean_absolute_error(y_test, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "r2": float(r2_score(y_test, pred)),
    }

# Pick best model from CV (lowest RMSE)
best_name = pd.DataFrame(results).sort_values("rmse").iloc[0]["model"]
best_est = models[best_name]
best_pipe = Pipeline([( "preprocess", preprocess), ("model", best_est)])

if has_student_list:
    train_mask = data["STUDENT_LIST"].astype(str).str.upper().eq("LIST15")
    test_mask = data["STUDENT_LIST"].astype(str).str.upper().eq("LIST16")

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
else:
    # Fallback: last fold as holdout (still group-based)
    splitter = GroupKFold(n_splits=5)
    splits = list(splitter.split(X, y, groups=groups))
    train_idx, test_idx = splits[-1]
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

holdout_metrics = evaluate_holdout(best_pipe, X_train, y_train, X_test, y_test)
best_name, holdout_metrics

('hgb',
 {'mae': 0.19355228463814353,
  'rmse': 0.32443884892514213,
  'r2': 0.7200992381354561})

In [ ]:
# Train final model on all data and save pipeline
import joblib

final_pipe = best_pipe.fit(X, y)

joblib.dump(
    {
        "pipeline": final_pipe,
        "features": features,
        "metrics_cv": results,
        "metrics_holdout": holdout_metrics,
    },
    OUT_DIR / "cgpa_model.joblib",
)

OUT_DIR / "cgpa_model.joblib"

WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/outputs/cgpa_model.joblib')

## Sample: test how the model works

The cell below loads the saved pipeline and runs a **sample prediction** on a few rows so you can see how the model is used in practice: same features in → predicted CGPA out.

In [ ]:
# Load saved model and feature list (run after the pipeline has been saved)
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

_OUT_DIR = Path.cwd() / "outputs"
_artifact = joblib.load(_OUT_DIR / "cgpa_model.joblib")
_sample_pipe = _artifact["pipeline"]
_sample_features = _artifact["features"]

# Use the same modeling table; take 5 rows as sample (e.g. first 5 with valid target)
_sample_df = pd.read_csv(_OUT_DIR / "modeling_table_student_semester.csv")
_sample_df = _sample_df.dropna(subset=["CGPA"]).head(5)
X_sample = _sample_df[_sample_features].copy()
y_actual = _sample_df["CGPA"].astype(float)

# Predict
y_pred = _sample_pipe.predict(X_sample)

# Show: REG_NO, SEMESTER_INDEX, actual CGPA, predicted CGPA, error
sample_result = _sample_df[["REG_NO", "SEMESTER_INDEX"]].copy()
sample_result["CGPA_actual"] = y_actual.values
sample_result["CGPA_predicted"] = np.round(y_pred, 3)
sample_result["error"] = np.round(y_pred - y_actual.values, 3)
print("Sample prediction (5 rows): same features in → predicted CGPA out.\n")
display(sample_result)

## Feature importance (permutation on holdout)

This converts the “deeper analytics” into a concrete *selected variable set* by ranking features by out-of-sample importance.

In [ ]:
from sklearn.inspection import permutation_importance

# Prepare holdout data from earlier cell (X_train/X_test/y_train/y_test)
final_pipe = best_pipe.fit(X_train, y_train)

perm = permutation_importance(
    final_pipe,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    scoring="neg_root_mean_squared_error",
)

imp = pd.DataFrame({
    "feature": X_test.columns,
    "importance_rmse": perm.importances_mean,
}).sort_values("importance_rmse", ascending=False)

imp.head(30)

,feature,importance_rmse
20,prev_cgpa,0.458709
34,progression_count,0.018110
0,PROGRAM,0.006019
11,total_paid_success,0.004728
5,present_rate,0.002580
9,payment_txn_count,0.001842
21,prev_semester_gpa,0.001519
6,late_rate,0.001296
30,prev_cum_credits_passed,0.001225
14,payment_span_days,0.001208


## Extra models: Neural network, XGBoost, and Random Forest

In this section we build three additional prediction models on the **same features** and **same train/test split**:

- A simple **neural network** (multi-layer perceptron) using past CGPA, attendance, payments, sponsorship, etc.
- An **XGBoost** model (gradient-boosted trees) that can capture complex patterns.
- A dedicated **Random Forest** model.

> Note: these models all use the payment features (like `total_paid_success` and `payment_span_days`), which reflect how quickly and how fully a student pays tuition. This connects directly to the rules you described: if tuition is not 75\% paid by weeks 6–7, coursework and exams are affected; if 100\% is not paid, the student may get `MEX`, which lowers CGPA.

In [ ]:
# EXTRA MODELS: Neural network, XGBoost, Random Forest
# -----------------------------------------------------
# This cell assumes you ALREADY ran the cells that create:
#   - X_train, y_train, X_test, y_test  (from the holdout split)
#   - preprocess (the ColumnTransformer)

from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor

try:
    import xgboost as xgb
    has_xgb = True
except ImportError:
    has_xgb = False
    print("xgboost is not installed. Run 'pip install xgboost' to enable the XGBoost model.\n")

# Helper function to train a model and print metrics in a simple way
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def train_and_report(name, estimator, X_train, y_train, X_test, y_test):
    """Fit a pipeline (preprocess + estimator) and print MAE, RMSE, and R^2.

    We keep the code simple so you can see:
    - We fit on training data only.
    - We evaluate on the test (holdout) data.
    """
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", estimator),
    ])

    print(f"\n=== {name} ===")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R^2  : {r2:.4f}")

    return pipe


# 1) Neural network (simple multi-layer perceptron)
#    This is a basic feed-forward network with two hidden layers.
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 32),  # two hidden layers: 64 and 32 neurons
    activation="relu",
    max_iter=50,                  # keep small for faster runs; increase for better accuracy
    random_state=0,
)

nn_pipe = train_and_report("Neural network (MLPRegressor)", mlp_reg, X_train, y_train, X_test, y_test)

# 2) Random Forest (separate from the earlier one, but conceptually similar)
rf_reg_extra = RandomForestRegressor(
    n_estimators=150,
    random_state=0,
    n_jobs=-1,
    min_samples_leaf=5,
)

rf_pipe_extra = train_and_report("Random Forest (extra run)", rf_reg_extra, X_train, y_train, X_test, y_test)

# 3) XGBoost (only if the library is available)
if has_xgb:
    xgb_reg = xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=0,
        n_jobs=-1,
    )

    xgb_pipe = train_and_report("XGBoost Regressor", xgb_reg, X_train, y_train, X_test, y_test)
else:
    xgb_pipe = None

print("\nDone training extra models.")

In [ ]:
# Save selected top features (you can tune k)
TOP_K = 25
selected_features = imp.head(TOP_K)["feature"].tolist()

pd.Series(selected_features, name="selected_feature").to_csv(OUT_DIR / "selected_features_topk.csv", index=False)
OUT_DIR / "selected_features_topk.csv"

WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/outputs/selected_features_topk.csv')